In [1]:
%matplotlib widget
# Boilerplate import code for all libraries
# Changes to the precision require re-loading the kernel and need to be done before any op uses them.
import sphWarpCore_config as swc
from typing import Any
swc.configure(precision="float32", dim=Any) # precision: float16|half|float32|single|float64|double

import sphWarpCore as sph
from sphWarpCore.type_config import *
print(get_type_config()) # confirms active settings

# Initialize warp at this point
import warp as wp; wp.init()

import os
import torch
if torch.cuda.is_available(): # set the TORCH_CUDA_ARCH_LIST environment variable to the compute capability of the GPU for faster compiles
    os.environ['TORCH_CUDA_ARCH_LIST'] = f'{torch.cuda.get_device_properties(0).major}.{torch.cuda.get_device_properties(0).minor}'

import warnings
from tqdm import TqdmExperimentalWarning
warnings.filterwarnings("ignore", category=TqdmExperimentalWarning)
from tqdm.autonotebook import tqdm

# final import blocks that are generic
import matplotlib.pyplot as plt
from torch.profiler import profile, record_function, ProfilerActivity
import numpy as np
import math
import shlex    
import subprocess
import shutil

# custom SPH libraries
from integrators.integration import *
from sphWarpCore import *

# This library
from compressibleSPH import *

{'scalar_t': <class 'warp._src.types.float32'>, 'dim_t': typing.Any}
Warp 1.12.0 initialized:
   CUDA Toolkit 12.9, Driver 13.2
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "NVIDIA RTX PRO 500 Blackwell Generation Laptop GPU" (6 GiB, sm_120, mempool enabled)
   Kernel cache:
     /home/lu26029/.cache/warp/1.12.0


In [2]:
import math
import numpy as np


nx = 256
dim = 2

L = 1
dx = L/nx
aspect = 2
band = 20

n_h = 4

gamma = 1.4
rho0 = 1.0

rho_I = 1.0
p_I = 1.0
rho_II = 0.125
p_II = 0.1
rho_III = 1.0
p_III = 0.1



extraData = {
    'nx': nx,
    'dim': dim,
    'L': L,
    'n_h': n_h,

    'gamma': gamma,
    'rho0': rho0,
    'rho_I': rho_I,
    'p_I': p_I,
    'rho_II': rho_II,
    'p_II': p_II,
    'rho_III': rho_III,
    'p_III': p_III,

    'dx': dx,
    'aspect': aspect,
    'band': band

}

In [3]:
device = torch.device('cuda:0') if torch.cuda.is_available() else torch.device('cpu')
dtype = get_torch_precision()


domain = buildDomainDescription(l = 1, dim = dim, periodic = True, device = device, dtype = dtype)
domain.min[0] = 0
domain.max[0] = 14
domain.min[1] = -3
domain.max[1] = 3

config, integrator = buildConfig(
    domain = domain,
    dim = dim,
    kernel = KernelFunctions.B7,
    targetNeighbors = n_h_to_nH(4, dim),
    supportMode = SupportScheme.KernelMeanSymmetric,
    gradientMode = GradientScheme.Difference,
    laplacianMode = LaplacianScheme.Brookshaw,
    integrationScheme = IntegrationSchemeType.rungeKutta2,
    samplingScheme = SamplingScheme.regular,
    device = device,
    dtype = dtype,
    dt = None,
    adaptiveDt = True,
    cflFactor=0.3,
)
config.nx = nx

config.minDt = 1e-8
# config.dx = L / (nx * 2)

scheme = CompressibleSPHScheme.CRKSPH
SimulationSystem, SimulationState, SimulationConfig, SimulationUpdate, fn, export_fn, import_fn = buildScheme(scheme)


schemeConfig = SimulationConfig()
schemeConfig.gamma = gamma
schemeConfig.rho0 = rho0


schemeConfig.viscositySwitchParams.scheme = ViscositySwitch.NoneSwitch
schemeConfig.adaptiveSupportScheme = AdaptiveSupportScheme.Owen
schemeConfig.adaptiveSupportCorrections = False

In [4]:
compressibleSystem = setupBasicCompressibleInitialState(nx, config, schemeConfig, SimulationState, SimulationSystem)
print(f"Number of particles: {compressibleSystem.state.positions.shape[0]}")

Number of particles: 153088


In [5]:
particles = compressibleSystem.state

rhoInitial = particles.densities.clone()
Pi = particles.pressures.clone()

regionI = torch.logical_or(particles.positions[:,0] <= 1.0, particles.positions[:,0] >= 13)
regionII = torch.logical_and(~regionI, particles.positions[:,1].abs() >= 1.5)
regionIII = torch.logical_and(~regionI, particles.positions[:,1].abs() < 1.5)

rhoInitial[regionI] = rho_I
Pi[regionI] = p_I
rhoInitial[regionII] = rho_II
Pi[regionII] = p_II
rhoInitial[regionIII] = rho_III
Pi[regionIII] = p_III

densityRatio = rhoInitial / particles.densities
# particles.masses = particles.masses * densityRatio


In [6]:
Lx = config.domain.max[0] - config.domain.min[0]
Ly = config.domain.max[1] - config.domain.min[1]
aspect = Ly / Lx

dx = Lx / (nx / aspect)
dy = Ly / nx

area = Lx * Ly / nx**2 * aspect


In [8]:
config.dx

tensor(0.0234, device='cuda:0')

In [7]:
# Pinitial = Pi
# rhoInitial = rho_y


A_, u_, P_, c_s = idealGasEOS(A = None, u = None, P = Pi, rho = rhoInitial, gamma = gamma)
# v_initial = torch.zeros_like(particles_l.positions)

vInitial = torch.zeros_like(particles.positions)
internalEnergy = u_ 
kineticEnergy = torch.linalg.norm(vInitial, dim = -1) **2/ 2
totalEnergy = (internalEnergy + kineticEnergy) * compressibleSystem.state.masses

compressibleSystem.state.internalEnergies = u_
compressibleSystem.state.totalEnergies = totalEnergy
compressibleSystem.state.pressures = P_
compressibleSystem.state.soundspeeds = c_s
compressibleSystem.state.velocities = vInitial
compressibleSystem.state.densities = rhoInitial
compressibleSystem.state.masses = area * rhoInitial


compressibleSPHConfigAdaptiveH = CompressibleSPHConfig(
    adaptiveSupportIterations=16,
    adaptiveSupportThreshold=1e-3,
    adaptiveSupportScheme=AdaptiveSupportScheme.Owen,
)

rho_optimal, h_optimal, adjacency, rhos_iter, supports_iter = evaluateOptimalSupport(compressibleSystem.state, config, supportScheme = SupportScheme.Gather, compParams = compressibleSPHConfigAdaptiveH)

compressibleSystem.state.supports = h_optimal
compressibleSystem.state.densities = rho_optimal

from compressibleSPH.modules.timestep.compressible import computeTimestep
config.dt = computeTimestep(compressibleSystem, config, schemeConfig, dt = None) #* 2/3
print(f"Computed timestep: {config.dt} {config.dt/0.0007333333293596903} s")

display(config)

Module compressibleSPH.modules.adaptiveSupport.wp_psi 8875fe6 load on device 'cuda:0' took 2.79 ms  (cached)
Module sphWarpCore.radiusSearch.wp_compactHash e67fccd load on device 'cuda:0' took 2.62 ms  (cached)
Module compressibleSPH.modules.adaptiveSupport.wp_psi0 37e5819 load on device 'cuda:0' took 7.96 ms  (cached)
Module sphWarpCore.operations.wp_density f0357bf load on device 'cuda:0' took 6.91 ms  (cached)
Computed timestep: 0.006195208057761192 8.448011033632598 s


SimulationConfig(device=device(type='cuda', index=0), dtype=torch.float32, domain=DomainDescription(min=tensor([ 0., -3.], device='cuda:0'), max=tensor([14.,  3.], device='cuda:0'), periodic=tensor([True, True], device='cuda:0'), dim=2), dim=2, verletScale=1.4142135623730951, kernel=<KernelFunctions.B7: 33>, integrationScheme=<IntegrationSchemeType.rungeKutta2: 1>, cflFactor=0.3, dt=0.006195208057761192, minDt=1e-08, maxDt=0.01, dtGrowthFactor=1.1, adaptiveDt=True, dx=tensor(0.0234, device='cuda:0'), nx=256, targetNeighbors=50.26548245743669, supportMode=<SupportScheme.KernelMeanSymmetric: 14>, gradientMode=<GradientScheme.Difference: 3>, laplacianMode=<LaplacianScheme.Brookshaw: 2>, samplingScheme=<SamplingScheme.regular: 1>)

In [18]:
runningState = compressibleSystem.initializeNewState()

kineticEnergy = 0.5 * (torch.linalg.norm(runningState.state.velocities, dim = -1) **2 * runningState.state.masses).sum()
thermalEnergy = (runningState.state.internalEnergies * runningState.state.masses).sum()
totalEnergy = kineticEnergy + thermalEnergy

In [28]:
caseName = '14-Triple_point'
exportPath = prepExport(f'{caseName}', config, schemeConfig, scheme, export_fn)
exportSimulationSystem(exportPath, 'initialState', scheme, compressibleSystem, exportAdjacency = False, stages = None, exportStagesAdjacency = False, extraData = dict({
    'kineticEnergy': kineticEnergy,
    'thermalEnergy': thermalEnergy,
    'totalEnergy': totalEnergy,
    'frame_num': 0,
}, **extraData))


In [31]:
from warpPlot import visualize, PlottingOptions, PlotScaling, GridVisualization, UniformColorMap, DivergingColorMap, Mapping, CyclicColorMap
markerSize = 1
plotter = visualize(
    particleState = runningState.state,
    domain = config.domain,
    quantities = {
        "A": runningState.state.densities,
    },
    plotOptions = {
        "A": PlottingOptions(
            colorMap = DivergingColorMap.RdBu,
            flipColorMap = True,
            markerSize = markerSize,
            midPoint = 0.0,
            quantityScaling = PlotScaling.Logarithmic,
            plotTitle = "density",
            gridVisualization = GridVisualization(
                resolution = 1024,
            ),
            vMin=0.2,
            vMax=7
        ),
    },
    figTitle = "Wave Equation Example",
    mosaic = 'A',
    figsize= (12,6),
    backend='vispy',
)

imagePath = f'{exportPath}/images'
os.makedirs(imagePath, exist_ok = True)
plotter.export(f'{imagePath}/frame_00000.png', dpi = 300)
# if args.exportImages:

RFBOutputContext()

In [ ]:
# config.dt = 2.5e-3
t_limit = 10.0
nSteps = int(t_limit / config.dt)

print(f"Running with dt: {config.dt}, which gives nSteps: {nSteps}")
# nSteps = 256

runningState = compressibleSystem.initializeNewState()

trajectory = []

priorStep = None
i = 0
t = 0
tq = tqdm(total = 1000, leave = True)

while t < t_limit:

    begin = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    begin.record()
    result = integrator.function(
        state = runningState,
        f = fn,
        dt = config.dt,  
        config = config,
        compParams = schemeConfig,
        verbose = False,
        # priorStep = priorStep
    )
    end.record()
    torch.cuda.synchronize()
    priorStep = result.stages[-1]
    timing = begin.elapsed_time(end)

    runningState = result.state
    kineticEnergy = 0.5 * (torch.linalg.norm(runningState.state.velocities, dim = -1) **2 * runningState.state.masses).sum()
    thermalEnergy = (runningState.state.internalEnergies * runningState.state.masses).sum()
    totalEnergy = kineticEnergy + thermalEnergy

    trajectory.append(
        (i, (i+1)*config.dt, totalEnergy.item(), kineticEnergy.item(), thermalEnergy.item(), timing)
,     )
    config.dt = computeTimestep(runningState, config, schemeConfig, dt = config.dt) #* 2/3

    i = i + 1
    t = runningState.t

    if i % 10 == 0 and i > 0:
        plotter.updateQuantities(
            {
                "A": runningState.state.velocities,
                "B": runningState.state.densities,
            },
            newParticleState = runningState.state,
        )
        plotter.export(f'{imagePath}/frame_{i:05d}.png', dpi = 300)
        
    if i % 500 == 0:
        exportSimulationSystem(exportPath, f'state_{i:04d}', scheme, runningState, exportAdjacency = False, stages = result.stages, exportStagesAdjacency = True, extraData = dict(**extraData, **{
            'kineticEnergy': kineticEnergy,
            'thermalEnergy': thermalEnergy,
            'totalEnergy': totalEnergy,
            'frame_num': i,
        }))

        
    maxVel = torch.linalg.norm(runningState.state.velocities, dim = -1).max()
    tq.set_description(f"Step {i+1}/{nSteps}, time: {(i+1)*config.dt:8.4g}/{t_limit:8.4g}, TE: {totalEnergy:.3g}, KE: {kineticEnergy:.3g}, IE: {thermalEnergy:.3g} | max vel: {maxVel:.3g} | iter time: {timing:.3f} ms")
    # t = {runningState.t:2f}, dt = {config.dt:.3g}, ptcls = {len(runningState.state.positions)}\nTotal Energy: {totalEnergy:.3g}, Kinetic Energy: {kineticEnergy:.3g}, Thermal Energy: {thermalEnergy:.3g}'
    # break

Running with dt: 0.00025, which gives nSteps: 20000


  0%|          | 0/1000 [00:00<?, ?it/s]

In [ ]:
result = integrator.function(
    state = runningState,
    f = fn,
    dt = config.dt,
    config = config,
    compParams = compressibleSPHConfig,
    verbose = False,
    priorStep = priorStep
)

densityRatio = runningState.state.densities / result.state.state.densities

if (densityRatio < 0.7).any() or (densityRatio > 1.5).any():
    print(f"Warning: Density ratio out of bounds at step {i}, min: {densityRatio.min().item()}, max: {densityRatio.max().item()}")

print(runningState.state.densities.max(), runningState.state.densities.min())
print(result.state.state.densities.max(), result.state.state.densities.min())

In [ ]:
from integrators.euler import updateStateEuler

halfState = runningState.initializeNewState()
halfState = updateStateEuler(halfState, priorStep.update, config.dt * 0.5, False)

In [ ]:
# halfState.adjacency = None

In [ ]:
config.verletScale

In [ ]:
stepResult = fn(halfState, config.dt * 0.5, config, compressibleSPHConfig)

In [ ]:
nnrsPrev = runningState.adjacency.numNeighbors
print(f'Number of neighbors: min {nnrsPrev.min().item()}, max {nnrsPrev.max().item()}, mean {nnrsPrev.to(torch.float32).mean().item()}')
nnrsCurr = stepResult[1].numNeighbors
print(f'Number of neighbors: min {nnrsCurr.min().item()}, max {nnrsCurr.max().item()}, mean {nnrsCurr.to(torch.float32).mean().item()}')

In [ ]:
fig, axis = plt.subplots(1, 2, figsize=(11,10), squeeze=False)
sc = axis[0,0].scatter(runningState.state.positions[:,0].cpu(), runningState.state.positions[:,1].cpu(), c = nnrsPrev.cpu(), s = 1, cmap = 'viridis')
fig.colorbar(sc, ax = axis[0,0])
sc = axis[0,1].scatter(stepResult[2].positions[:,0].cpu(), stepResult[2].positions[:,1].cpu(), c = nnrsCurr.cpu(), s = 1, cmap = 'viridis')
fig.colorbar(sc, ax = axis[0,1])
for ax in axis.flatten():
    ax.set_aspect('equal')
    ax.set_xlim(config.domain.min[0].item(), config.domain.max[0].item())
    ax.set_ylim(config.domain.min[1].item(), config.domain.max[1].item())
    ax.set_xlabel('x')
    ax.set_ylabel('y')
fig.tight_layout()

In [ ]:
fig, axis = plt.subplots(1, 2, figsize=(11,10), squeeze=False)
sc = axis[0,0].scatter(runningState.state.positions[:,0].cpu(), runningState.state.positions[:,1].cpu(), c = runningState.state.densities.cpu(), s = 1, cmap = 'viridis')
fig.colorbar(sc, ax = axis[0,0])
sc = axis[0,1].scatter(stepResult[2].positions[:,0].cpu(), stepResult[2].positions[:,1].cpu(), c = stepResult[2].densities.cpu(), s = 1, cmap = 'viridis')
fig.colorbar(sc, ax = axis[0,1])
for ax in axis.flatten():
    ax.set_aspect('equal')
    ax.set_xlim(config.domain.min[0].item(), config.domain.max[0].item())
    ax.set_ylim(config.domain.min[1].item(), config.domain.max[1].item())
    ax.set_xlabel('x')
    ax.set_ylabel('y')
fig.tight_layout()

In [ ]:
fig, axis = plt.subplots(1, 2, figsize=(11,10), squeeze=False)
sc = axis[0,0].scatter(runningState.state.positions[:,0].cpu(), runningState.state.positions[:,1].cpu(), c = runningState.state.supports.cpu(), s = 1, cmap = 'viridis')
fig.colorbar(sc, ax = axis[0,0])
sc = axis[0,1].scatter(stepResult[2].positions[:,0].cpu(), stepResult[2].positions[:,1].cpu(), c = stepResult[2].supports.cpu(), s = 1, cmap = 'viridis')
fig.colorbar(sc, ax = axis[0,1])
for ax in axis.flatten():
    ax.set_aspect('equal')
    ax.set_xlim(config.domain.min[0].item(), config.domain.max[0].item())
    ax.set_ylim(config.domain.min[1].item(), config.domain.max[1].item())
    ax.set_xlabel('x')
    ax.set_ylabel('y')
fig.tight_layout()

In [ ]:
import copy
queryPositions = halfState.state.positions
referencePositions = halfState.state.positions
querySupports = halfState.state.supports
referenceSupports = halfState.state.supports
domain = copy.deepcopy(config.domain)
domain.min = domain.min
domain.max = domain.max

verletScale = 1.0
mode = SupportScheme.Scatter

adjacency, hmap = radiusSearchCompactHashMap_(
    queryPositions, referencePositions,
    querySupports * verletScale, referenceSupports * verletScale,
    domain.periodic, domain, mode, hashMapLength=queryPositions.shape[0] + 1, returnCompactHashMap = True
)

print(adjacency.numNeighbors.min(), adjacency.numNeighbors.max(), adjacency.numNeighbors.to(torch.float).mean())

In [ ]:

print(torch.sum(adjacency.i == adjacency.j), queryPositions.shape[0])

In [ ]:
print(hmap.hashTable)
print(hmap.sortedCellTable.shape)

In [ ]:

queryPositions = halfState.state.positions.to(torch.float64)
referencePositions = halfState.state.positions.to(torch.float64)
querySupports = halfState.state.supports.to(torch.float64)
referenceSupports = halfState.state.supports.to(torch.float64)
domain = copy.deepcopy(config.domain)
domain.min = domain.min.to(torch.float64)
domain.max = domain.max.to(torch.float64)

In [ ]:
domainDescription = config.domain
periodicity = domainDescription.periodic

mode_uint = supportSchemeToUint(mode)
# mode_map = {'gather': 1, 'scatter': 2, 'symmetric': 3, 'superSymmetric': 4, ''}
# mode_uint = mode_map.get(mode, 0)
# if mode_uint == 0:
    # raise ValueError(f"Invalid mode: {mode}. Supported modes are: {list(mode_map.keys())}")
    
minDomain = domainDescription.min if domainDescription.min is not None else None
maxDomain = domainDescription.max if domainDescription.max is not None else None
hMax = computeGridSupport(querySupports, referenceSupports, mode)
minD, maxD = getDomainExtents(referencePositions, minDomain, maxDomain)
x = torch.vstack([component if not periodic else torch.remainder(component - minD[i], maxD[i] - minD[i]) + minD[i] for i, (component, periodic) in enumerate(zip(referencePositions.mT, periodicity))]).mT
y = torch.vstack([component if not periodic else torch.remainder(component - minD[i], maxD[i] - minD[i]) + minD[i] for i, (component, periodic) in enumerate(zip(queryPositions.mT, periodicity))]).mT

sortedLinear, sortIndex, numCells, qMin, qMax, hCell = sortReferenceParticles(x, hMax, minD, maxD)


sortedPositions = x[sortIndex,:]

cellIndices, cellCounters = torch.unique_consecutive(sortedLinear, return_counts=True, return_inverse=False)
cellCounters = cellCounters.to(torch.int32)
# Needs to zero padded for the indexing to work properly as the 0th cell is valid and cumsum doesn't include the first element

cumCell = torch.hstack((torch.tensor([0], device = cellIndices.device, dtype=cellCounters.dtype),torch.cumsum(cellCounters,dim=0)))[:-1].to(torch.int32)

sortedIndices = torch.floor((sortedPositions - qMin) / to_numpy(hCell)).to(torch.int32)
for d in range(sortedIndices.shape[1]):
    sortedIndices[:, d] = torch.clamp(sortedIndices[:, d], 0, int(numCells[d].item()) - 1)
cellGridIndices = sortedIndices[cumCell,:]
cellTable = torch.stack((cellIndices, cumCell, cellCounters), dim = 1)

warpDevice = castTorchToWarp(queryPositions).device
torchDevice = queryPositions.device

cellGridIndices_warp = castTorchToWarp(cellGridIndices)
hashedIndices_warp = wp.zeros(sortedIndices.shape[0], dtype=wp.uint32, device=warpDevice)
wp.launch(hashCells, dim=sortedIndices.shape[0], inputs=[sortedIndices, wp.uint32(2), hashedIndices_warp], device=warpDevice)
hashedIndices = wp.to_torch(hashedIndices_warp).to(torch.int32)

In [ ]:
fig, axis = plt.subplots(1, 2, figsize=(11,10), squeeze=False)

q = sortedLinear
qUnique = torch.unique(q)
colors = torch.rand((qUnique.shape[0], 3), device = queryPositions.device)
c = colors[torch.searchsorted(qUnique, q)]

s = 1

sc = axis[0,0].scatter(sortedPositions[:,0].cpu(), sortedPositions[:,1].cpu(), c = c.cpu(), s = s, cmap = 'viridis')
# fig.colorbar(sc, ax = axis[0,0])

q = hashedIndices
qUnique = torch.unique(q)
colors = torch.rand((qUnique.shape[0], 3), device = queryPositions.device)
c = colors[torch.searchsorted(qUnique, q)]

sc = axis[0,1].scatter(sortedPositions[:,0].cpu(), sortedPositions[:,1].cpu(), c = c.cpu(), s = s, cmap = 'viridis')
# fig.colorbar(sc, ax = axis[0,1])
for ax in axis.flatten():
    ax.set_aspect('equal')
    ax.set_xlim(config.domain.min[0].item(), config.domain.max[0].item())
    ax.set_ylim(config.domain.min[1].item(), config.domain.max[1].item())
    ax.set_xlabel('x')
    ax.set_ylabel('y')

In [ ]:
ffmpeg -framerate 50 -f image2 -pattern_type glob -i 'frame_*.png' -c:v libx264 -pix_fmt yuv420p -b:v 10M output.mp4
ffmpeg -i output.mp4  -vf "fps=50,scale=540:-1:flags=lanczos,palettegen" palette.png
ffmpeg -i output.mp4 -i palette.png -filter_complex "fps=25,scale=540:-1:flags=lanczos[x];[x][1:v]paletteuse" out.gif

In [ ]:
cState = runningState.initializeNewState()
currentState = cState.state


adjacency = buildVerletList(
    currentState, 
    config.domain, verletScale = 1.0, supportMode = SupportScheme.SuperSymmetric,
    priorNeighborhood = None,
    verbose = False)

apparentVolume, currentState.densities, crkState = computeCRKFactors(currentState, config.domain, config.kernel, adjacency = adjacency)


In [ ]:
def limiterVL(x):
    # if x <= 0.0:
        # return 0.0
    # x = wp.min(x, scalar_t(1.0e6))
    # vL = 2.0 / (1.0 + x)
    # return x * vL*vL

    return (x + torch.abs(x)) / (1.0 + torch.abs(x))

xx = torch.linspace(0, 1, 1000)
yy = limiterVL(xx)

fig, ax = plt.subplots()
ax.plot(xx.cpu(), yy.cpu())
ax.set_title("Van Leer Limiter")
ax.set_xlabel("x")
ax.set_ylabel("limiterVL(x)")
plt.grid()


In [ ]:
velocityGradient = warpOperation(
    currentState,
    OperationProperties(
        kernel = config.kernel,
        operation = WarpOperation.Gradient,
        supportMode = SupportScheme.Scatter, # E.3
        gradientMode = GradientScheme.Difference, # E.3
    ),
    queryValues = currentState.velocities,
    domain = config.domain,
    adjacency = adjacency,
    # queryVolumes = apparentVolume,
    # crkState= crkState,
)

velocityGradientCRK = warpOperation(
    currentState,
    OperationProperties(
        kernel = config.kernel,
        operation = WarpOperation.Gradient,
        supportMode = SupportScheme.Scatter, # E.3
        gradientMode = GradientScheme.Difference, # E.3
    ),
    queryValues = currentState.velocities,
    domain = config.domain,
    adjacency = adjacency,
    queryVolumes = apparentVolume,
    crkState= crkState,
)

In [ ]:
trace = torch.einsum('...ii', velocityGradient)

traces = torch.eye(velocityGradient.shape[1], device=velocityGradient.device) * trace.view(-1, 1, 1) / velocityGradient.shape[1]

Shear = (velocityGradient + velocityGradient.transpose(1,2))/2 - traces
Rotation = (velocityGradient - velocityGradient.transpose(1,2)) / 2

In [ ]:
matrix = velocityGradient# - velocityGradientCRK
from warpPlot import visualize, PlottingOptions, PlotScaling, GridVisualization, UniformColorMap, DivergingColorMap, Mapping
markerSize = 2
plotter = visualize(
    particleState = currentState,
    domain = config.domain,
    quantities = {
        "A": matrix[:,0,0],
        "B": matrix[:,0,1],
        "C": matrix[:,1,0],
        "D": matrix[:,1,1],
    },
    plotOptions = {
        "A": PlottingOptions(
            colorMap = UniformColorMap.viridis,
            markerSize = markerSize,
            midPoint = 0.0,
            quantityScaling = PlotScaling.Linear,
        ),
        "B": PlottingOptions(
            colorMap = UniformColorMap.viridis,
            markerSize = markerSize,
            midPoint = 0.0,
            quantityScaling = PlotScaling.Linear,
        ),
        "C": PlottingOptions(
            colorMap = UniformColorMap.viridis,
            markerSize = markerSize,
            midPoint = 0.0,
            quantityScaling = PlotScaling.Linear,
        ),
        "D": PlottingOptions(
            colorMap = UniformColorMap.viridis,
            markerSize = markerSize,
            midPoint = 0.0,
            quantityScaling = PlotScaling.Linear,
        ),
    },
    figTitle = "Wave Equation Example",
    mosaic = '''AB
    CD''',
    figsize= (11,10),
    # backend='vispy',
    # backend='pyVista',
    # backendOptions = {
    #     # In notebooks, use trame for reliable live updates.
    #     'jupyter_backend': 'trame',
    # }
)

# if args.exportImages:
#     plotter.export(f'output/{folderName}/frame_00000.png', dpi = args.figureDpi)

In [ ]:
display(plotter.fig)